In [ ]:
%load_ext autoreload
%autoreload 2
import dt4dds_benchmark
import plotly.express as px
import pandas as pd

data = dt4dds_benchmark.analysis.Dataset.combine(*[dt4dds_benchmark.pipelines.HDF5Manager(f'./data/{s}.hdf5').get_data() for s in (
    'aeon_high',
    'aeon_low',
    'aeon_medium',
    'fountain_high',
    'fountain_low',
    'fountain_medium',
    'goldman_default',
    'rs_high',
    'rs_low',
    'rs_medium',
    'hedges_low',
    'hedges_medium',
    'yinyang_default',
)])

### check fit of simulation results

In [ ]:
for c in data.separate_by_parameters(['codec.type', 'codec.name', 'clustering.name', 'clustering.type']):
    c.fit('workflow.dropout').plot(title_columns=['codec.type', 'codec.name', 'clustering.name', 'clustering.type']).show()

### get the threshold values by codec, clustering, and scenario

In [ ]:
df = data.get_fits_by_group(['codec.type', 'codec.name', 'clustering.name', 'clustering.type'], on='workflow.dropout', additional_agg={'code_rate': 'mean'})
df['code_rate'] = df['code_rate'].map('{:.2f}'.format)

df

### plot dropout

In [ ]:
plotdf = df.copy()
plotdf['id'] = plotdf['codec.type'] + '_' + plotdf['codec.name'] + '_' + plotdf['clustering.type']
plotdf['codec.type'] = plotdf['codec.type'].str.replace('YinYang', 'YY').replace('Goldman', 'GM')
plotdf = plotdf.loc[plotdf['clustering.type'] != 'BasicSet'].copy()

plotdf

In [ ]:
fig = dt4dds_benchmark.analysis.plotting.tiered_bar(
    plotdf.sort_values(['codec.type', 'code_rate']),
    "codec.type",
    "code_rate",
    "threshold",
)
fig.update_yaxes(
    title_text='Dropout',
    tickformat=",.0%",
    dtick=0.2,
)
fig.update_layout(
    width=320,
    height=120,
    margin=dict(l=0, r=2, t=2, b=30),
    showlegend=False,
)


fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
fig.update_xaxes(
    tickfont_size=28/3, 
    tickangle=0,
)
fig.show()
fig.write_image(f'./figures/individual_plot.svg')

# save data
plotdf.to_csv('./figures/individual_plot.csv', index=False)

### Check duration and memory constraints

In [ ]:
performancedf = pd.merge(data.combined_performances, data.results)
# performancedf = performancedf.drop(performancedf.loc[performancedf['decoding_success'] == False].index)
performancedf = performancedf.drop(performancedf.loc[performancedf['identifier'] != b'decoding'].index)
performancedf['code_rate'] = performancedf['code_rate'].map('{:.2f}'.format)
performancedf.loc[performancedf['code_rate'] == '1.51', 'code_rate'] = '1.50'
performancedf.loc[performancedf['code_rate'] == '1.01', 'code_rate'] = '1.00'
performancedf.loc[performancedf['clustering.type'] == 'BasicSet', 'clustering.type'] = 'Naive'
performancedf.loc[performancedf['clustering.type'] != 'Naive', 'clustering.type'] = 'Clustering'

for codec in performancedf['codec.type'].unique():
    idf = performancedf.loc[(performancedf['codec.type'] == codec)].copy()
    idf['codec.order'] = idf['codec.name'].map({'high': 0, 'medium': 1, 'low': 2, 'default': 3})
    idf = idf.sort_values(['codec.order', 'clustering.type'], ascending=False)
    idf['duration'] = idf['duration'] / 60
    idf['name'] = idf['code_rate'] + " bit/nt"

    fig = px.scatter(
        idf,
        x='workflow.dropout',
        y='duration',
        facet_col='name',
        facet_row='clustering.type',
        facet_row_spacing=0.2,
        facet_col_spacing=0.075,
        symbol='decoding_success',
        symbol_map={True: 'circle', False: 'circle-open'},
    )
    fig.add_hline(y=60, line_dash="dot", line_color="black", line_width=2)
    fig.update_xaxes(matches=None)
    fig.for_each_xaxis(lambda xaxis: xaxis.update(range=[0, 0.8], minor_dtick=0.1, dtick=0.2))
    fig.for_each_yaxis(lambda yaxis: yaxis.update(range=[0, 61], minor_dtick=10))
    fig.update_xaxes(title='Dropout', row=1)
    fig.update_yaxes(title='Runtime / min', col=1)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_layout(
        width=315,
        height=200,
        margin=dict(l=0, r=10, t=20, b=10),
        showlegend=False,
    )
    fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
    fig.show()
    fig.write_image(f'./figures/duration_{codec}.svg')

    # save data
    idf.to_csv(f'./figures/duration_{codec}.csv', index=False)

    fig = px.scatter(
        idf,
        x='workflow.dropout',
        y='memory_value',
        facet_col='name',
        facet_row='clustering.type',
        facet_row_spacing=0.2,
        facet_col_spacing=0.075,
        symbol='decoding_success',
        color_discrete_map={'overall': '#636363', 'substitution': '#de2d26', 'insertion': '#31a354', 'deletion': '#3182bd'},
        symbol_map={True: 'circle', False: 'circle-open'},
    )
    fig.add_hline(y=8, line_dash="dot", line_color="black", line_width=2)
    fig.update_xaxes(matches=None)
    fig.for_each_xaxis(lambda xaxis: xaxis.update(range=[0, 0.8], minor_dtick=0.1, dtick=0.2))
    fig.for_each_yaxis(lambda yaxis: yaxis.update(range=[0, 8.1], minor_dtick=1))
    fig.update_xaxes(title='Dropout', row=1)
    fig.update_yaxes(title='Memory / GB', col=1)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_layout(
        width=315,
        height=200,
        margin=dict(l=0, r=10, t=20, b=10),
        showlegend=False,
    )
    fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
    fig.show()
    fig.write_image(f'./figures/memory_{codec}.svg')

    # save data
    idf.to_csv(f'./figures/memory_{codec}.csv', index=False)